In [1]:

!pip install -U pandas numpy scikit-learn sentencepiece accelerate transformers datasets protobuf tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 1.5 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 1.2 MB/s  0:00:14m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 1.1 MB/s  0:00:08m0:00:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 1.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 1.5 MB/s  0:00:07m0:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 1.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 1.8 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 1.6 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 1.5 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 2.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48

In [ ]:
from pathlib import Path
import pandas as pd

for p in [Path("/workspace/sents.tsv"), Path("/workspace/words.tsv"), Path("/workspace/data/sents.tsv"), Path("/workspace/data/words.tsv")]:
    if p.exists():
        print(p, "size =", p.stat().st_size)

def try_read(path):
    for enc in ["utf-8-sig", "utf-8", "cp1251", "latin1"]:
        try:
            df = pd.read_csv(path, sep="\t", encoding=enc, on_bad_lines="skip")
            print("\nSUCCESS:", path, "encoding =", enc)
            print("columns:", df.columns.tolist())
            print(df.head(3))
            return df
        except Exception as e:
            print("\nFAILED:", path, "encoding =", enc, "|", type(e).__name__, e)
    return None

try_read("/workspace/sents.tsv")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import re
import random
import warnings
import shutil
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
    TrainerCallback,
    EarlyStoppingCallback,
)

# ── Reproducibility ─────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Config ──────────────────────────────────────────────────────────────────────
def find_data_dir():
    for d in [Path("/workspace/data"), Path("/workspace")]:
        if (d / "sents.tsv").exists() and (d / "words.tsv").exists():
            return d
    raise FileNotFoundError("Could not find sents.tsv and words.tsv in /workspace or /workspace/data")

DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/workspace/outputs/mt5_bg_gec_stage2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/mt5-base"
MAX_LENGTH = 160
MAX_PAIRS = 150_000
IDENTITY_RATIO = 0.12

PREFIX = "fix grammar: "

BATCH_SIZE = 8
GRAD_ACCUM = 1
LR = 2e-4
WEIGHT_DECAY = 0.01
EPOCHS = 4
PATIENCE = 2

print("=" * 80)
print(f"CUDA available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device         : {torch.cuda.get_device_name(0)}")
    print(f"GPU memory     : {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB free / {torch.cuda.mem_get_info()[1] / 1e9:.2f} GB total")
    print(f"BF16 supported  : {torch.cuda.is_bf16_supported()}")
print("=" * 80)
print(f"DATA_DIR       : {DATA_DIR}")

# ── Helpers ────────────────────────────────────────────────────────────────────
def normalize_bg(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r"\s+([.,!?;:%)\]])", r"\1", text)
    text = re.sub(r"([(\[«“])\s+", r"\1", text)
    text = re.sub(r"\s+-\s+", "-", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def last_alpha(word: str) -> str:
    for ch in reversed(word):
        if ch.isalpha():
            return ch.lower()
    return ""

def is_consonant(ch: str) -> bool:
    return ch.lower() in set("бвгджзйклмнпрстфхцчшщ")

def replace_nth(text, original, replacement, n=0):
    pattern = r'(?<!\w)' + re.escape(original) + r'(?!\w)'
    matches = list(re.finditer(pattern, text))
    if not matches or n >= len(matches):
        return None
    m = matches[n]
    return text[:m.start()] + replacement + text[m.end():]

def occurrence_index(tokens, target_idx):
    w = tokens[target_idx]["word"]
    return sum(1 for t in tokens[:target_idx] if t["word"] == w)

def read_tsv_flexible(path: Path):
    encodings = ["utf-8-sig", "utf-8", "cp1251", "latin1"]
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, sep="\t", encoding=enc, on_bad_lines="skip")
        except Exception as e:
            last_err = e
    raise last_err

def load_sents(path: Path):
    df = read_tsv_flexible(path)
    cols = [c.lower().strip() for c in df.columns.tolist()]
    col_map = {c.lower().strip(): c for c in df.columns.tolist()}

    if "text" in cols:
        text_col = col_map["text"]
        id_col = col_map["sent_id"] if "sent_id" in cols else (col_map["id"] if "id" in cols else None)
        if id_col is None:
            df = df.reset_index().rename(columns={"index": "sent_id"})
            id_col = "sent_id"
        out = df[[id_col, text_col]].copy()
        out.columns = ["sent_id", "text"]
        return out

    if "sentence" in cols:
        text_col = col_map["sentence"]
        id_col = col_map["id"] if "id" in cols else None
        if id_col is None:
            df = df.reset_index().rename(columns={"index": "sent_id"})
            id_col = "sent_id"
        out = df[[id_col, text_col]].copy()
        out.columns = ["sent_id", "text"]
        return out

    if df.shape[1] == 2:
        df = df.copy()
        df.columns = ["sent_id", "text"]
        return df

    raise ValueError(f"Could not infer sents.tsv columns: {df.columns.tolist()}")

def load_words(path: Path):
    df = read_tsv_flexible(path)
    cols = [c.lower().strip() for c in df.columns.tolist()]
    col_map = {c.lower().strip(): c for c in df.columns.tolist()}

    aliases = {
        "sent_id": ["sent_id", "id", "sen"],
        "word": ["word", "w"],
        "lemma": ["lemma", "l"],
        "pos": ["pos", "tag"]
    }
    pick = {}
    for out_name, names in aliases.items():
        found = None
        for n in names:
            if n in cols:
                found = col_map[n]
                break
        if found is None:
            raise ValueError(f"Could not infer words.tsv column for '{out_name}'. Found: {df.columns.tolist()}")
        pick[out_name] = found

    out = df[[pick["sent_id"], pick["word"], pick["lemma"], pick["pos"]]].copy()
    out.columns = ["sent_id", "word", "lemma", "pos"]
    return out

# ── Load data ───────────────────────────────────────────────────────────────────
print("\n[1/8] Loading data...")

sents = load_sents(DATA_DIR / "sents.tsv")
words = load_words(DATA_DIR / "words.tsv")

sents["text"] = sents["text"].map(normalize_bg)
words["word"] = words["word"].map(normalize_bg)

print(f"  Sentences : {len(sents):,}")
print(f"  Tokens    : {len(words):,}")

# ── Filter ─────────────────────────────────────────────────────────────────────
print("\n[2/8] Filtering sentences...")

sents = sents[sents["text"].str.len() <= 300].reset_index(drop=True)

has_verb = set(words.loc[words["pos"].astype(str).str.startswith("V", na=False), "sent_id"])
word_counts = words.groupby("sent_id").size()
valid_ids = set(word_counts[word_counts >= 3].index) & has_verb

sents_f = sents[sents["sent_id"].isin(valid_ids)].reset_index(drop=True)
print(f"  Valid sentences : {len(sents_f):,}")

# ── Index tokens ───────────────────────────────────────────────────────────────
print("\n[3/8] Indexing tokens...")

words = words[words["sent_id"].isin(valid_ids)].copy()
words["pos_in_sent"] = words.groupby("sent_id").cumcount()

sent_tokens = (
    words.sort_values(["sent_id", "pos_in_sent"])
    .groupby("sent_id")
    .apply(lambda g: g[["pos_in_sent", "word", "lemma", "pos"]].to_dict("records"))
    .to_dict()
)

del words
gc.collect()
print(f"  Indexed {len(sent_tokens):,} sentences")

# ── Error induction rules ──────────────────────────────────────────────────────
print("\n[4/8] Defining error induction rules...")

ARTICLE_SWAPS = [("ът", "а"), ("ят", "я"), ("та", "а"), ("то", "о")]

def induce_article_misuse(tokens, text):
    cands = []
    for i, t in enumerate(tokens):
        if not str(t["pos"]).startswith("N"):
            continue
        w = t["word"]
        for suf, repl in ARTICLE_SWAPS:
            if w.endswith(suf) and len(w) > len(suf) + 2:
                cands.append((i, w, w[:-len(suf)] + repl))
                break
    if not cands:
        return None, None
    i, orig, err = random.choice(cands)
    return replace_nth(text, orig, err, occurrence_index(tokens, i)), "article_misuse"

def article_suffix_for(word):
    lw = last_alpha(word)
    if lw in {"а", "я"}:
        return "та"
    if lw in {"о", "е", "ю"}:
        return "то"
    if is_consonant(lw):
        return random.choice(["ът", "ят"])
    return None

def induce_article_insertion(tokens, text):
    cands = []
    for i, t in enumerate(tokens):
        if not str(t["pos"]).startswith("N"):
            continue
        w = t["word"]
        if any(w.endswith(suf) for suf, _ in ARTICLE_SWAPS):
            continue
        suf = article_suffix_for(w)
        if suf and len(w) > 2:
            cands.append((i, w, w + suf))
    if not cands:
        return None, None
    i, orig, err = random.choice(cands)
    return replace_nth(text, orig, err, occurrence_index(tokens, i)), "article_insertion"

PRONOUN_MAP = {
    "който": "когото", "когото": "който", "което": "когото", "които": "когото",
    "чийто": "чиито", "чиито": "чийто",
    "той": "него", "него": "той", "тя": "нея", "нея": "тя",
    "те": "тях", "тях": "те", "ние": "нас", "нас": "ние",
    "вие": "вас", "вас": "вие",
}

def induce_pronoun_misuse(tokens, text):
    cands = []
    for i, t in enumerate(tokens):
        if not str(t["pos"]).startswith(("P", "T")):
            continue
        lw = t["word"].lower()
        if lw in PRONOUN_MAP:
            cands.append((i, t["word"], PRONOUN_MAP[lw]))
    if not cands:
        return None, None
    i, orig, repl = random.choice(cands)
    if orig and orig[0].isupper():
        repl = repl[0].upper() + repl[1:]
    return replace_nth(text, orig, repl, occurrence_index(tokens, i)), "pronoun_misuse"

REL_PRONOUN_MAP = {
    "който": ["която", "което", "които"],
    "която": ["който", "което", "които"],
    "което": ["който", "която", "които"],
    "които": ["който", "която", "което"],
}

def induce_relative_pronoun_misuse(tokens, text):
    cands = []
    for i, t in enumerate(tokens):
        lw = t["word"].lower()
        if lw in REL_PRONOUN_MAP:
            cands.append((i, t["word"], random.choice(REL_PRONOUN_MAP[lw])))
    if not cands:
        return None, None
    i, orig, repl = random.choice(cands)
    if orig and orig[0].isupper():
        repl = repl[0].upper() + repl[1:]
    return replace_nth(text, orig, repl, occurrence_index(tokens, i)), "relative_pronoun_misuse"

def induce_verb_suffix(tokens, text):
    cands = [
        (i, t) for i, t in enumerate(tokens)
        if str(t["pos"]).startswith("V") and t["word"].endswith("м") and len(t["word"]) > 2
    ]
    if not cands:
        return None, None
    i, t = random.choice(cands)
    err = replace_nth(text, t["word"], t["word"] + "е", occurrence_index(tokens, i))
    return err, "incorrect_verb_suffix"

VERB_AGREE_MAP = {
    "е": "са", "са": "е",
    "беше": "бяха", "бяха": "беше",
    "съм": "сме", "сме": "съм",
    "си": "сте", "сте": "си",
    "бях": "бяхме", "бяхме": "бях",
}

PLURAL_HINTS = {"те", "ние", "вие", "хора", "деца", "жени", "мъже", "ученици", "учители"}

def induce_verb_agreement(tokens, text):
    cands = []
    token_words = [t["word"].lower() for t in tokens]
    pluralish = any(w in PLURAL_HINTS for w in token_words)

    for i, t in enumerate(tokens):
        lw = t["word"].lower()
        if lw in VERB_AGREE_MAP:
            if pluralish or random.random() < 0.4:
                cands.append((i, t["word"], VERB_AGREE_MAP[lw]))
    if not cands:
        return None, None
    i, orig, repl = random.choice(cands)
    if orig and orig[0].isupper():
        repl = repl[0].upper() + repl[1:]
    return replace_nth(text, orig, repl, occurrence_index(tokens, i)), "verb_agreement"

ADJ_SWAPS = [
    ("ия", "ата"), ("ата", "ия"), ("ото", "ия"),
    ("ите", "ия"), ("ият", "ия"), ("ия", "ият"),
    ("ото", "а"), ("ата", "а"),
]

def induce_noun_adj_disagree(tokens, text):
    cands = []
    for i, t in enumerate(tokens):
        if not str(t["pos"]).startswith("A"):
            continue
        has_noun_nb = (
            (i > 0 and str(tokens[i - 1]["pos"]).startswith("N")) or
            (i < len(tokens) - 1 and str(tokens[i + 1]["pos"]).startswith("N"))
        )
        if not has_noun_nb:
            continue
        w = t["word"]
        for suf, repl in ADJ_SWAPS:
            if w.endswith(suf) and len(w) > len(suf) + 1:
                cands.append((i, w, w[:-len(suf)] + repl))
                break
    if not cands:
        return None, None
    i, orig, err = random.choice(cands)
    return replace_nth(text, orig, err, occurrence_index(tokens, i)), "noun_adj_disagree"

def induce_predicative_adj_agreement(tokens, text):
    cands = []
    for i, t in enumerate(tokens):
        if not str(t["pos"]).startswith("A"):
            continue
        w = t["word"]
        # very simple endings to force more adjective-agreement variety
        variants = []
        if len(w) > 3 and is_consonant(last_alpha(w)):
            variants = [w + "а", w + "о", w + "и"]
        elif w.endswith("а") or w.endswith("о") or w.endswith("и"):
            stem = w[:-1]
            variants = [stem, stem + "а", stem + "о", stem + "и"]
        else:
            continue
        variants = [v for v in variants if v and v != w]
        if variants:
            cands.append((i, w, random.choice(variants)))
    if not cands:
        return None, None
    i, orig, err = random.choice(cands)
    return replace_nth(text, orig, err, occurrence_index(tokens, i)), "predicative_adj_agreement"

PREP_SWAP = {
    "в": "на", "на": "в",
    "от": "до", "до": "от",
    "за": "към", "към": "за",
    "с": "със", "със": "с",
    "при": "пред", "пред": "при",
    "по": "в", "в": "по",
}

def induce_preposition_swap(tokens, text):
    cands = []
    for i, t in enumerate(tokens):
        lw = t["word"].lower()
        if lw in PREP_SWAP and len(t["word"]) <= 4:
            cands.append((i, t["word"], PREP_SWAP[lw]))
    if not cands:
        return None, None
    i, orig, repl = random.choice(cands)
    if orig and orig[0].isupper():
        repl = repl[0].upper() + repl[1:]
    return replace_nth(text, orig, repl, occurrence_index(tokens, i)), "preposition_swap"

NOUN_SUFFIX_SHIFT = [
    ("а", "и"),
    ("я", "и"),
    ("о", "а"),
    ("е", "а"),
    ("ът", "а"),
    ("ят", "а"),
    ("и", "а"),
]

def induce_noun_inflection_shift(tokens, text):
    cands = []
    for i, t in enumerate(tokens):
        if not str(t["pos"]).startswith("N"):
            continue
        w = t["word"]
        for suf, repl in NOUN_SUFFIX_SHIFT:
            if w.endswith(suf) and len(w) > len(suf) + 2:
                cands.append((i, w, w[:-len(suf)] + repl))
                break
    if not cands:
        return None, None
    i, orig, err = random.choice(cands)
    return replace_nth(text, orig, err, occurrence_index(tokens, i)), "noun_inflection_shift"

INDUCERS = [
    induce_article_misuse,
    induce_article_insertion,
    induce_pronoun_misuse,
    induce_relative_pronoun_misuse,
    induce_verb_suffix,
    induce_verb_agreement,
    induce_noun_adj_disagree,
    induce_predicative_adj_agreement,
    induce_preposition_swap,
    induce_noun_inflection_shift,
]

def induce_error(tokens, text):
    funcs = INDUCERS.copy()
    random.shuffle(funcs)
    for fn in funcs:
        err, etype = fn(tokens, text)
        if err and err != text:
            return err, etype
    return None, None

# ── Generate pairs ─────────────────────────────────────────────────────────────
print(f"\n[5/8] Generating up to {MAX_PAIRS:,} error pairs...")

pairs = []
for _, row in sents_f.iterrows():
    if len(pairs) >= MAX_PAIRS:
        break
    sid = row["sent_id"]
    text = normalize_bg(row["text"])
    if sid not in sent_tokens:
        continue

    if random.random() < IDENTITY_RATIO:
        pairs.append({
            "correct": text,
            "erroneous": text,
            "error_type": "identity"
        })

    err, etype = induce_error(sent_tokens[sid], text)
    if err and err != text:
        pairs.append({
            "correct": text,
            "erroneous": normalize_bg(err),
            "error_type": etype
        })

pairs = pairs[:MAX_PAIRS]
df = pd.DataFrame(pairs).drop_duplicates().reset_index(drop=True)

del sent_tokens, sents_f
gc.collect()

print(f"  Generated : {len(df):,} pairs")
print(df["error_type"].value_counts().to_string())
assert len(df) > 0, "No pairs generated."

# ── Split ───────────────────────────────────────────────────────────────────────
def safe_split(dataframe, test_size, seed, stratify_col=None):
    if stratify_col is not None:
        counts = dataframe[stratify_col].value_counts()
        if len(counts) > 1 and counts.min() >= 2:
            return train_test_split(
                dataframe,
                test_size=test_size,
                random_state=seed,
                stratify=dataframe[stratify_col],
            )
    return train_test_split(
        dataframe,
        test_size=test_size,
        random_state=seed,
    )

train_df, temp_df = safe_split(df, test_size=0.28, seed=SEED, stratify_col="error_type")
val_df, test_df   = safe_split(temp_df, test_size=0.357, seed=SEED, stratify_col="error_type")

train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "val.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)

print(f"\n  Train={len(train_df):,}  Val={len(val_df):,}  Test={len(test_df):,}")

# ── Tokenizer / Dataset ────────────────────────────────────────────────────────
print("\n[6/8] Building tokenizer and datasets...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class GECDataset(Dataset):
    def __init__(self, df, tok, max_len=MAX_LENGTH, prefix=PREFIX):
        self.data = df.reset_index(drop=True)
        self.tok = tok
        self.max_len = max_len
        self.prefix = prefix

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        src = normalize_bg(row["erroneous"])
        tgt = normalize_bg(row["correct"])

        enc = self.tok(
            self.prefix + src,
            max_length=self.max_len,
            truncation=True,
            padding=False,
        )
        lab = self.tok(
            text_target=tgt,
            max_length=self.max_len,
            truncation=True,
            padding=False,
        )
        enc["labels"] = lab["input_ids"] or [self.tok.eos_token_id]
        return enc

train_ds = GECDataset(train_df, tokenizer)
val_ds   = GECDataset(val_df, tokenizer)
test_ds  = GECDataset(test_df, tokenizer)

sample = train_ds[0]
print("  Sample source:", tokenizer.decode(sample["input_ids"], skip_special_tokens=True))
print("  Sample target:", tokenizer.decode(sample["labels"], skip_special_tokens=True))

# ── Model ──────────────────────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

print(f"\n  GPU free before model load: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

print("\n[7/8] Loading model...")

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,
)
model.config.use_cache = False
model = model.to("cuda:0")

print(f"  GPU free after model load: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB")

model.train()
dummy_inputs = tokenizer("fix grammar: тест изречение", return_tensors="pt", truncation=True).to("cuda:0")
dummy_labels = tokenizer(text_target="тест изречение", return_tensors="pt")["input_ids"].to("cuda:0")
dummy_loss = model(**dummy_inputs, labels=dummy_labels).loss
print(f"  Dummy loss: {dummy_loss.item():.4f}")
assert not torch.isnan(dummy_loss), "Dummy loss is NaN"

model.zero_grad(set_to_none=True)
del dummy_inputs, dummy_labels, dummy_loss
torch.cuda.empty_cache()

print(f"\n  Testing BATCH_SIZE={BATCH_SIZE} ...")
preview_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

try:
    test_batch = [train_ds[i] for i in range(min(BATCH_SIZE, len(train_ds)))]
    test_collated = preview_collator(test_batch)
    test_collated = {k: v.to("cuda:0") for k, v in test_collated.items()}
    out = model(**test_collated)
    out.loss.backward()
    model.zero_grad(set_to_none=True)
    peak_gb = torch.cuda.max_memory_allocated() / 1e9
    print(f"  Batch OK — peak memory: {peak_gb:.2f} GB")
    torch.cuda.reset_peak_memory_stats()
except torch.cuda.OutOfMemoryError:
    print("  OOM with current batch size; falling back to batch=4, grad_accum=2")
    BATCH_SIZE = 4
    GRAD_ACCUM = 2
    torch.cuda.empty_cache()
    gc.collect()

# ── Callback ───────────────────────────────────────────────────────────────────
class NanLossCallback(TrainerCallback):
    def __init__(self):
        self.zero_count = 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        loss = logs.get("loss")
        if loss is None:
            return
        if loss != loss:
            print("\n[NanLossCallback] Loss is NaN — stopping.")
            control.should_training_stop = True
        elif loss == 0.0:
            self.zero_count += 1
            if self.zero_count >= 3:
                print("\n[NanLossCallback] Loss stuck at 0 — stopping.")
                control.should_training_stop = True
        else:
            self.zero_count = 0

# ── Train ──────────────────────────────────────────────────────────────────────
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=0,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    predict_with_generate=False,
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),
    optim="adafactor",
    logging_steps=50,
    report_to="none",
    label_smoothing_factor=0.0,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    remove_unused_columns=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[NanLossCallback(), EarlyStoppingCallback(early_stopping_patience=PATIENCE)],
)

print("\nStarting training...\n")
trainer.train()

# ── Evaluation ─────────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("POST-TRAINING CHECK")
print("=" * 70)

model.eval()

print(f"  shared std = {model.shared.weight.std().item():.4f}")
print(f"  lm_head std = {model.lm_head.weight.std().item():.4f}")
print(f"  equal tensors = {torch.equal(model.shared.weight, model.lm_head.weight)}")

def batch_generate(texts, batch_size=8, use_model=None):
    use_model = use_model or model
    results = []
    for i in range(0, len(texts), batch_size):
        batch = [normalize_bg(t) for t in texts[i:i + batch_size]]
        inputs = tokenizer(
            [PREFIX + t for t in batch],
            return_tensors="pt",
            max_length=MAX_LENGTH,
            truncation=True,
            padding=True,
        ).to("cuda:0")
        with torch.no_grad():
            out = use_model.generate(
                **inputs,
                max_new_tokens=MAX_LENGTH,
                num_beams=4,
            )
        results.extend([normalize_bg(x) for x in tokenizer.batch_decode(out, skip_special_tokens=True)])
    return results

def compute_f05_strict(preds, refs, srcs):
    TP = FP = FN = 0
    for pred, ref, src in zip(preds, refs, srcs):
        pred = normalize_bg(pred)
        ref  = normalize_bg(ref)
        src  = normalize_bg(src)

        pred_toks = pred.strip().split()
        ref_toks  = ref.strip().split()
        src_toks  = src.strip().split()
        min_len   = min(len(src_toks), len(ref_toks), len(pred_toks))

        found_error = False
        for j in range(min_len):
            if src_toks[j] != ref_toks[j]:
                found_error = True
                if pred_toks[j] == ref_toks[j]:
                    TP += 1
                else:
                    FN += 1
                    if pred_toks[j] != src_toks[j]:
                        FP += 1

        if not found_error and len(ref_toks) != len(src_toks):
            FN += abs(len(ref_toks) - len(src_toks))
        if len(pred_toks) != len(ref_toks) and len(src_toks) == len(ref_toks):
            FP += abs(len(pred_toks) - len(ref_toks))

    prec = TP / (TP + FP) * 100 if (TP + FP) > 0 else 0
    rec  = TP / (TP + FN) * 100 if (TP + FN) > 0 else 0
    f05  = (1.25 * prec * rec) / (0.25 * prec + rec + 1e-9)
    return prec, rec, f05, TP, FP, FN

sources    = [normalize_bg(PREFIX + s) for s in test_df["erroneous"].tolist()]
references = [normalize_bg(s) for s in test_df["correct"].tolist()]
erroneous  = [normalize_bg(s) for s in test_df["erroneous"].tolist()]
etypes     = test_df["error_type"].tolist()

predictions = batch_generate(sources)

prec, rec, f05, TP, FP, FN = compute_f05_strict(predictions, references, erroneous)

print(f"\n{'='*60}")
print(f"  Strict edit-only evaluation")
print(f"  TP={TP}  FP={FP}  FN={FN}")
print(f"  Precision : {prec:.2f}%")
print(f"  Recall    : {rec:.2f}%")
print(f"  F0.5      : {f05:.2f}%")
print(f"{'='*60}")

print("\nPer error type:")
for etype in sorted(test_df["error_type"].unique()):
    mask  = [e == etype for e in etypes]
    p_sub = [p for p, m in zip(predictions, mask) if m]
    r_sub = [r for r, m in zip(references, mask) if m]
    s_sub = [s for s, m in zip(erroneous, mask) if m]
    ep, er, ef, etp, efp, efn = compute_f05_strict(p_sub, r_sub, s_sub)
    print(f"  {etype:<28} P={ep:.1f}%  R={er:.1f}%  F0.5={ef:.1f}%  TP={etp} FP={efp} FN={efn}  n={sum(mask)}")

print(f"\n{'='*60}")
print("Sample corrections:")
print(f"{'='*60}")

sample_n = min(15, len(test_df))
sample_df = test_df.sample(sample_n, random_state=7)
sample_preds = batch_generate([PREFIX + s for s in sample_df["erroneous"].tolist()])

for row, pred in zip(sample_df.itertuples(), sample_preds):
    correct = "✓" if normalize_bg(pred) == normalize_bg(row.correct) else "✗"
    print(f"  [{correct}] {row.error_type}")
    print(f"      ERRONEOUS : {row.erroneous}")
    print(f"      PREDICTED : {pred}")
    print(f"      CORRECT   : {row.correct}")
    print()

# ── Save ───────────────────────────────────────────────────────────────────────
best_path = OUTPUT_DIR / "best_model_stage2"
best_path.mkdir(parents=True, exist_ok=True)

model.config.tie_word_embeddings = False
model.save_pretrained(best_path, safe_serialization=False)
tokenizer.save_pretrained(best_path)

print(f"\nModel saved to: {best_path}")

zip_base = str(OUTPUT_DIR / "mt5_bg_gec_stage2_model")
shutil.make_archive(zip_base, "zip", str(best_path))
print(f"Zipped to: {zip_base}.zip")

print("\nManual spot-check:")
manual_tests = [
    "Жената е красив.",
    "Книгата на маса е интересна.",
    "Той отиде на пазара да купи хляб.",
    "че според тя няма как иначе",
    "развитиео на проекта",
    "клеткаа е важна",
    "когото ги е направил",
    "Голямата планина, в който се качихме, беше голяма.",
]
manual_out = batch_generate(manual_tests)
for src, out in zip(manual_tests, manual_out):
    print(f"IN : {src}")
    print(f"OUT: {out}")
    print()